# 模型架構

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random

# ----------------------------
# 1. 時間序列資料集介面
# ----------------------------
class TimeSeriesDataset(Dataset):
    """
    時間序列資料集介面，資料格式預設為 numpy array 或 torch.Tensor，形狀為 (N, channels, series_length)
    """
    def __init__(self, data):
        """
        熊熊，請確認 data 為 numpy array 或 torch.Tensor，並且形狀為 (N, channels, series_length)
        """
        if isinstance(data, np.ndarray):
            self.data = torch.from_numpy(data)
        else:
            self.data = data
    
    def __len__(self):
        return self.data.shape[0]
    
    def __getitem__(self, idx):
        return self.data[idx]

# ----------------------------
# 2. 時間序列增強函數 (例如 jittering)
# ----------------------------
def time_series_augmentation(x, noise_std=0.01):
    """
    對輸入的時間序列 x 進行簡單的 jittering 增強：加上均值為 0、標準差為 noise_std 的噪聲
    x: Tensor，形狀 (batch, channels, series_length)
    """
    noise = torch.randn_like(x) * noise_std
    return x + noise

# ----------------------------
# 3. NT-Xent 對比損失函數 (參考 TS-TCC 論文)
# ----------------------------
def nt_xent_loss(z1, z2, temperature=0.5):
    """
    計算 NT-Xent 損失
    z1, z2: 形狀 (batch_size, latent_dim) 的隱向量
    """
    batch_size = z1.shape[0]
    # 將向量正規化
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    
    # 拼接成一個 2N x latent_dim 的矩陣
    representations = torch.cat([z1, z2], dim=0)  # shape: (2*batch_size, latent_dim)
    
    # 計算相似度矩陣，使用餘弦相似度（內積已經是正規化後的餘弦相似度）
    similarity_matrix = torch.matmul(representations, representations.T)  # shape: (2N, 2N)
    
    # 除以溫度參數並取指數
    sim_exp = torch.exp(similarity_matrix / temperature)
    
    # 建立遮罩，排除相同樣本（對角線）
    mask = (~torch.eye(2 * batch_size, 2 * batch_size, dtype=bool, device=z1.device)).float()
    
    # 每個向量的分母（排除自己）
    denom = torch.sum(sim_exp * mask, dim=1)
    
    # 正向樣本：對於 i 和 i+batch_size 為正向對
    pos_sim = torch.exp(torch.sum(z1 * z2, dim=1) / temperature)
    pos_sim = torch.cat([pos_sim, pos_sim], dim=0)
    
    loss = -torch.log(pos_sim / denom)
    return loss.mean()

# ----------------------------
# 4. Generator 網路 (針對時間序列生成)
# ----------------------------
class TimeSeriesGenerator(nn.Module):
    def __init__(self, latent_dim=100, series_length=128, channels=1):
        """
        Generator 將潛在向量映射為時間序列數據
        假設 series_length 可被 4 整除，此處使用線性層配合 ConvTranspose1d 進行上採樣
        """
        super(TimeSeriesGenerator, self).__init__()
        self.latent_dim = latent_dim
        self.series_length = series_length
        self.channels = channels
        self.hidden_dim = 128  # 中間維度
        
        # 線性層：將 latent vector 映射至一個展平向量，準備 reshape 為 (batch, hidden_dim, series_length/4)
        self.fc = nn.Linear(latent_dim, self.hidden_dim * (series_length // 4))
        
        # 使用兩層 ConvTranspose1d 進行上採樣至原始長度
        self.deconv = nn.Sequential(
            nn.ConvTranspose1d(self.hidden_dim, self.hidden_dim // 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(self.hidden_dim // 2),
            nn.ReLU(True),
            nn.ConvTranspose1d(self.hidden_dim // 2, channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh()  # 輸出範圍 [-1, 1]
        )
    
    def forward(self, z):
        """
        z: 潛在向量，形狀 (batch, latent_dim)
        輸出: 生成的時間序列，形狀 (batch, channels, series_length)
        """
        x = self.fc(z)  # (batch, hidden_dim * (series_length/4))
        x = x.view(-1, self.hidden_dim, self.series_length // 4)
        x = self.deconv(x)
        return x

# ----------------------------
# 5. Critic 網路 (WGAN 中用於估計 Wasserstein 距離)
# ----------------------------
class TimeSeriesCritic(nn.Module):
    def __init__(self, series_length=128, channels=1):
        """
        Critic 使用 Conv1d 層將時間序列映射為一個標量分數
        """
        super(TimeSeriesCritic, self).__init__()
        self.series_length = series_length
        self.channels = channels
        self.hidden_dim = 128
        
        self.conv = nn.Sequential(
            nn.Conv1d(channels, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(64, self.hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.fc = nn.Linear(self.hidden_dim * (series_length // 4), 1)
    
    def forward(self, x):
        """
        x: 時間序列資料，形狀 (batch, channels, series_length)
        輸出: 每個樣本的 critic 分數，形狀 (batch, 1)
        """
        features = self.conv(x)
        features = features.view(features.size(0), -1)
        out = self.fc(features)
        return out

# ----------------------------
# 6. Encoder 網路 (依 TS-TCC 將時間序列映射至隱向量空間)
# ----------------------------
class TimeSeriesEncoder(nn.Module):
    def __init__(self, series_length=128, channels=1, latent_dim=100):
        """
        Encoder 結構與 Critic 類似，但最終輸出為 latent_dim 維度的隱向量
        """
        super(TimeSeriesEncoder, self).__init__()
        self.series_length = series_length
        self.channels = channels
        self.hidden_dim = 128
        
        self.conv = nn.Sequential(
            nn.Conv1d(channels, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, self.hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True)
        )
        self.fc = nn.Linear(self.hidden_dim * (series_length // 4), latent_dim)
    
    def forward(self, x):
        """
        x: 時間序列資料，形狀 (batch, channels, series_length)
        輸出: 隱向量表徵，形狀 (batch, latent_dim)
        """
        features = self.conv(x)
        features = features.view(features.size(0), -1)
        z = self.fc(features)
        return z

# ----------------------------
# 7. 整合 WGAN 與 TS-TCC 的模型類別
# ----------------------------
class TimeSeriesWGAN:
    def __init__(self, dataset, latent_dim=100, series_length=128, channels=1, batch_size=64, lr=1e-4,
                 weight_clip=0.01, critic_iter=5, device=None):
        """
        初始化模型，包含 Generator、Critic 與 Encoder
        - dataset: 時間序列資料集 (必須符合 TimeSeriesDataset 介面)
        - latent_dim: 潛在向量維度
        - series_length: 時間序列長度
        - channels: 時間序列通道數
        - batch_size: 批次大小
        - lr: 學習率
        - weight_clip: critic 權重截斷區間
        - critic_iter: 每次生成器更新前，critic 的更新次數
        - device: 運算設備
        """
        self.device = device if device is not None else ('cuda' if torch.cuda.is_available() else 'cpu')
        self.latent_dim = latent_dim
        self.series_length = series_length
        self.channels = channels
        self.batch_size = batch_size
        self.lr = lr
        self.weight_clip = weight_clip
        self.critic_iter = critic_iter
        
        # 初始化網路
        self.generator = TimeSeriesGenerator(latent_dim, series_length, channels).to(self.device)
        self.critic = TimeSeriesCritic(series_length, channels).to(self.device)
        self.encoder = TimeSeriesEncoder(series_length, channels, latent_dim).to(self.device)
        
        # 建立資料 DataLoader
        self.dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
        
        # 定義 optimizers
        self.optim_C = optim.RMSprop(self.critic.parameters(), lr=lr)  # 根據原 WGAN 論文，critic 常用 RMSprop
        self.optim_G = optim.RMSprop(self.generator.parameters(), lr=lr)
        # Encoder 可使用 Adam
        self.optim_E = optim.Adam(self.encoder.parameters(), lr=lr)
    
    def train(self, num_epochs=100):
        """
        模型訓練主流程，同時更新 Critic、Generator 與 Encoder (對比學習部分)
        """
        print(f"訓練開始，使用設備：{self.device}")
        for epoch in range(num_epochs):
            for i, real_data in enumerate(self.dataloader):
                real_data = real_data.to(self.device)
                batch_size = real_data.size(0)
                
                # ----------------------------
                # (1) 更新 Critic (多次更新)
                # ----------------------------
                for _ in range(self.critic_iter):
                    # Sample noise，生成假資料
                    z = torch.randn(batch_size, self.latent_dim, device=self.device)
                    fake_data = self.generator(z).detach()  # 停止梯度傳播給 Generator
                    
                    # Critic 輸出
                    critic_real = self.critic(real_data)
                    critic_fake = self.critic(fake_data)
                    
                    # 計算 critic loss
                    loss_C = -torch.mean(critic_real) + torch.mean(critic_fake)
                    
                    # 更新 critic
                    self.optim_C.zero_grad()
                    loss_C.backward()
                    self.optim_C.step()
                    
                    # 權重截斷，確保 Lipschitz 條件
                    for p in self.critic.parameters():
                        p.data.clamp_(-self.weight_clip, self.weight_clip)
                
                # ----------------------------
                # (2) 更新 Generator
                # ----------------------------
                z = torch.randn(batch_size, self.latent_dim, device=self.device)
                fake_data = self.generator(z)
                # Generator 目標：使 critic 對 fake_data 輸出更低的分數（因為 critic loss 為 critic_fake 越高越不利於 generator）
                loss_G = -torch.mean(self.critic(fake_data))
                
                self.optim_G.zero_grad()
                loss_G.backward()
                self.optim_G.step()
                
                # ----------------------------
                # (3) 更新 Encoder (對比學習)
                # ----------------------------
                # 針對每個真實資料，產生兩個增強版本
                aug1 = time_series_augmentation(real_data)
                aug2 = time_series_augmentation(real_data)
                
                # 透過 encoder 得到隱向量
                z1 = self.encoder(aug1)
                z2 = self.encoder(aug2)
                
                loss_E = nt_xent_loss(z1, z2, temperature=0.5)
                
                self.optim_E.zero_grad()
                loss_E.backward()
                self.optim_E.step()
                
                if i % 50 == 0:
                    print(f"[Epoch {epoch+1}/{num_epochs}] [批次 {i}/{len(self.dataloader)}] "
                          f"[Loss_C: {loss_C.item():.4f}] [Loss_G: {loss_G.item():.4f}] [Loss_E: {loss_E.item():.4f}]")
            # 結束一個 epoch 後可加入驗證或儲存模型的步驟
        
        print("訓練完成。")
    
    def generate(self, n_samples):
        """
        透過 Generator 生成 n_samples 筆時間序列資料
        """
        self.generator.eval()
        with torch.no_grad():
            z = torch.randn(n_samples, self.latent_dim, device=self.device)
            samples = self.generator(z)
        self.generator.train()
        return samples.cpu()
    
    def get_latent(self, x):
        """
        傳入 x（時間序列資料），透過 Encoder 得到隱向量表徵
        """
        self.encoder.eval()
        with torch.no_grad():
            latent = self.encoder(x.to(self.device))
        self.encoder.train()
        return latent.cpu()
    
    def get_model(self):
        """
        返回模型各元件：Generator, Critic, Encoder
        """
        return {
            "generator": self.generator,
            "critic": self.critic,
            "encoder": self.encoder
        }

# ----------------------------
# 8. 模型建構介面函數
# ----------------------------
def build_time_series_model(dataset, latent_dim=100, series_length=128, channels=1, batch_size=64, lr=1e-4,
                            weight_clip=0.01, critic_iter=5, device=None):
    """
    熊熊，請利用此介面帶入您的時間序列資料集，並建立一個完整的模型
    返回一個 TimeSeriesWGAN 實例，可進行訓練、生成樣本及取得隱向量表徵
    """
    model = TimeSeriesWGAN(dataset, latent_dim=latent_dim, series_length=series_length, channels=channels,
                           batch_size=batch_size, lr=lr, weight_clip=weight_clip, critic_iter=critic_iter,
                           device=device)
    return model

# ----------------------------
# 9. 測試執行 (可依需求移除或修改)
# ----------------------------
if __name__ == '__main__':
    # 產生一個 dummy 資料集，形狀為 (1000, 1, 128)
    dummy_data = np.random.randn(1000, 1, 128).astype(np.float32)
    dataset = TimeSeriesDataset(dummy_data)
    
    # 建立模型
    model = build_time_series_model(dataset, latent_dim=100, series_length=128, channels=1, batch_size=64, lr=1e-4)
    
    # 訓練模型 (這裡僅示範 10 個 epoch)
    model.train(num_epochs=100)
    
    # 生成 10 筆時間序列樣本
    gen_samples = model.generate(10)
    print("生成樣本的形狀：", gen_samples.shape)


訓練開始，使用設備：cuda
[Epoch 1/100] [批次 0/15] [Loss_C: -0.0018] [Loss_G: -0.0129] [Loss_E: 4.0404]
[Epoch 2/100] [批次 0/15] [Loss_C: -0.0132] [Loss_G: -0.0342] [Loss_E: 3.0216]
[Epoch 3/100] [批次 0/15] [Loss_C: -0.0071] [Loss_G: -0.0410] [Loss_E: 2.9588]
[Epoch 4/100] [批次 0/15] [Loss_C: -0.0219] [Loss_G: -0.0996] [Loss_E: 2.9405]
[Epoch 5/100] [批次 0/15] [Loss_C: -0.0269] [Loss_G: -0.1241] [Loss_E: 2.9413]
[Epoch 6/100] [批次 0/15] [Loss_C: -0.0415] [Loss_G: -0.1585] [Loss_E: 2.9540]
[Epoch 7/100] [批次 0/15] [Loss_C: -0.0448] [Loss_G: -0.1961] [Loss_E: 2.9249]
[Epoch 8/100] [批次 0/15] [Loss_C: -0.0381] [Loss_G: -0.2087] [Loss_E: 2.9314]
[Epoch 9/100] [批次 0/15] [Loss_C: -0.0439] [Loss_G: -0.2053] [Loss_E: 2.9432]
[Epoch 10/100] [批次 0/15] [Loss_C: -0.0438] [Loss_G: -0.2074] [Loss_E: 2.9503]
[Epoch 11/100] [批次 0/15] [Loss_C: -0.0382] [Loss_G: -0.2200] [Loss_E: 2.9498]
[Epoch 12/100] [批次 0/15] [Loss_C: -0.0469] [Loss_G: -0.2158] [Loss_E: 2.9506]
[Epoch 13/100] [批次 0/15] [Loss_C: -0.0443] [Loss_G: -0.209

# RLHF + Contrastive Learning

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import random
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import tkinter as tk
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

#############################################
# 1. 時間序列資料集介面
#############################################
class TimeSeriesDataset(Dataset):
    def __init__(self, data):
        """
        熊熊，請確認 data 為 numpy array 或 torch.Tensor，形狀為 (N, channels, series_length)
        """
        if isinstance(data, np.ndarray):
            self.data = torch.from_numpy(data)
        else:
            self.data = data
    
    def __len__(self):
        return self.data.shape[0]
    
    def __getitem__(self, idx):
        return self.data[idx]

#############################################
# 2. 時間序列增強函數 (例如 jittering)
#############################################
def time_series_augmentation(x, noise_std=0.01):
    noise = torch.randn_like(x) * noise_std
    return x + noise

#############################################
# 3. NT-Xent 對比損失函數
#############################################
def nt_xent_loss(z1, z2, temperature=0.5):
    batch_size = z1.shape[0]
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    
    representations = torch.cat([z1, z2], dim=0)  # (2N, latent_dim)
    similarity_matrix = torch.matmul(representations, representations.T)
    sim_exp = torch.exp(similarity_matrix / temperature)
    mask = (~torch.eye(2 * batch_size, 2 * batch_size, dtype=bool, device=z1.device)).float()
    denom = torch.sum(sim_exp * mask, dim=1)
    pos_sim = torch.exp(torch.sum(z1 * z2, dim=1) / temperature)
    pos_sim = torch.cat([pos_sim, pos_sim], dim=0)
    
    loss = -torch.log(pos_sim / denom + 1e-8)
    return loss.mean()

#############################################
# 4. Generator 網路 (針對時間序列生成)
#############################################
class TimeSeriesGenerator(nn.Module):
    def __init__(self, latent_dim=100, series_length=128, channels=1):
        super(TimeSeriesGenerator, self).__init__()
        self.latent_dim = latent_dim
        self.series_length = series_length
        self.channels = channels
        self.hidden_dim = 128
        
        self.fc = nn.Linear(latent_dim, self.hidden_dim * (series_length // 4))
        self.deconv = nn.Sequential(
            nn.ConvTranspose1d(self.hidden_dim, self.hidden_dim // 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(self.hidden_dim // 2),
            nn.ReLU(True),
            nn.ConvTranspose1d(self.hidden_dim // 2, channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )
    
    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, self.hidden_dim, self.series_length // 4)
        x = self.deconv(x)
        return x

#############################################
# 5. Critic 網路 (WGAN 中用於估計 Wasserstein 距離)
#############################################
class TimeSeriesCritic(nn.Module):
    def __init__(self, series_length=128, channels=1):
        super(TimeSeriesCritic, self).__init__()
        self.series_length = series_length
        self.channels = channels
        self.hidden_dim = 128
        
        self.conv = nn.Sequential(
            nn.Conv1d(channels, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(64, self.hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.fc = nn.Linear(self.hidden_dim * (series_length // 4), 1)
    
    def forward(self, x):
        features = self.conv(x)
        features = features.view(features.size(0), -1)
        out = self.fc(features)
        return out

#############################################
# 6. Encoder 網路 (依 TS-TCC 將時間序列映射至隱向量空間)
#############################################
class TimeSeriesEncoder(nn.Module):
    def __init__(self, series_length=128, channels=1, latent_dim=100):
        super(TimeSeriesEncoder, self).__init__()
        self.series_length = series_length
        self.channels = channels
        self.hidden_dim = 128
        
        self.conv = nn.Sequential(
            nn.Conv1d(channels, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, self.hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True)
        )
        self.fc = nn.Linear(self.hidden_dim * (series_length // 4), latent_dim)
    
    def forward(self, x):
        features = self.conv(x)
        features = features.view(features.size(0), -1)
        z = self.fc(features)
        return z

#############################################
# 7. 整合 WGAN 與 TS-TCC 的模型類別
#############################################
class TimeSeriesWGAN:
    def __init__(self, dataset, latent_dim=100, series_length=128, channels=1, batch_size=64, lr=1e-4,
                 weight_clip=0.01, critic_iter=5, device=None):
        self.device = device if device is not None else ('cuda' if torch.cuda.is_available() else 'cpu')
        self.latent_dim = latent_dim
        self.series_length = series_length
        self.channels = channels
        self.batch_size = batch_size
        self.lr = lr
        self.weight_clip = weight_clip
        self.critic_iter = critic_iter
        
        self.generator = TimeSeriesGenerator(latent_dim, series_length, channels).to(self.device)
        self.critic = TimeSeriesCritic(series_length, channels).to(self.device)
        self.encoder = TimeSeriesEncoder(series_length, channels, latent_dim).to(self.device)
        
        self.dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
        
        self.optim_C = optim.RMSprop(self.critic.parameters(), lr=lr)
        self.optim_G = optim.RMSprop(self.generator.parameters(), lr=lr)
        self.optim_E = optim.Adam(self.encoder.parameters(), lr=lr)
    
    def train(self, num_epochs=100):
        print(f"訓練開始，使用設備：{self.device}")
        for epoch in range(num_epochs):
            for i, real_data in enumerate(self.dataloader):
                real_data = real_data.to(self.device)
                batch_size = real_data.size(0)
                
                for _ in range(self.critic_iter):
                    z = torch.randn(batch_size, self.latent_dim, device=self.device)
                    fake_data = self.generator(z).detach()
                    
                    critic_real = self.critic(real_data)
                    critic_fake = self.critic(fake_data)
                    
                    loss_C = -torch.mean(critic_real) + torch.mean(critic_fake)
                    
                    self.optim_C.zero_grad()
                    loss_C.backward()
                    self.optim_C.step()
                    
                    for p in self.critic.parameters():
                        p.data.clamp_(-self.weight_clip, self.weight_clip)
                
                z = torch.randn(batch_size, self.latent_dim, device=self.device)
                fake_data = self.generator(z)
                loss_G = -torch.mean(self.critic(fake_data))
                
                self.optim_G.zero_grad()
                loss_G.backward()
                self.optim_G.step()
                
                aug1 = time_series_augmentation(real_data)
                aug2 = time_series_augmentation(real_data)
                
                z1 = self.encoder(aug1)
                z2 = self.encoder(aug2)
                
                loss_E = nt_xent_loss(z1, z2, temperature=0.5)
                
                self.optim_E.zero_grad()
                loss_E.backward()
                self.optim_E.step()
                
                if i % 50 == 0:
                    print(f"[Epoch {epoch+1}/{num_epochs}] [批次 {i}/{len(self.dataloader)}] "
                          f"[Loss_C: {loss_C.item():.4f}] [Loss_G: {loss_G.item():.4f}] [Loss_E: {loss_E.item():.4f}]")
        print("訓練完成。")
    
    def generate(self, n_samples):
        self.generator.eval()
        with torch.no_grad():
            z = torch.randn(n_samples, self.latent_dim, device=self.device)
            samples = self.generator(z)
        self.generator.train()
        return samples.cpu()
    
    def get_latent(self, x):
        self.encoder.eval()
        with torch.no_grad():
            latent = self.encoder(x.to(self.device))
        self.encoder.train()
        return latent.cpu()
    
    def get_model(self):
        return {
            "generator": self.generator,
            "critic": self.critic,
            "encoder": self.encoder
        }

#############################################
# 8. 模型建構介面函數
#############################################
def build_time_series_model(dataset, latent_dim=100, series_length=128, channels=1, batch_size=64, lr=1e-4,
                            weight_clip=0.01, critic_iter=5, device=None):
    model = TimeSeriesWGAN(dataset, latent_dim=latent_dim, series_length=series_length, channels=channels,
                           batch_size=batch_size, lr=lr, weight_clip=weight_clip, critic_iter=critic_iter,
                           device=device)
    return model

#############################################
# 9. Ranking Loss 與 RLHF 微調函數
#############################################
def ranking_loss(critic, preferred, non_preferred):
    score_pref = critic(preferred)
    score_nonpref = critic(non_preferred)
    loss = -torch.log(torch.sigmoid(score_pref - score_nonpref) + 1e-8)
    return loss.mean()

def rlhf_finetune(model, human_feedback, num_epochs=5, lambda_factor=0.5):
    """
    熊熊，進行 RLHF 微調：
    - model: TimeSeriesWGAN 實例
    - human_feedback: list of dict，每筆包含 "sample1", "sample2", "preferred"
    - num_epochs: 微調的 epoch 數
    - lambda_factor: RL 與對比損失權重
    """
    device = model.device
    optimizer = optim.Adam(list(model.generator.parameters()) + 
                           list(model.critic.parameters()) + 
                           list(model.encoder.parameters()), lr=model.lr)
    
    for epoch in range(num_epochs):
        total_loss = 0.0
        for fb in human_feedback:
            # 為避免梯度版本問題，先 detach 並 clone
            sample1 = fb["sample1"].detach().clone().to(device).unsqueeze(0)
            sample2 = fb["sample2"].detach().clone().to(device).unsqueeze(0)
            if fb["preferred"] == 1:
                pos_sample = sample1
                neg_sample = sample2
            elif fb["preferred"] == 2:
                pos_sample = sample2
                neg_sample = sample1
            else:
                raise ValueError("preferred 欄位必須為 1 或 2")
            
            # 增強正樣本
            pos_aug = time_series_augmentation(pos_sample)
            z_pos = model.encoder(pos_sample)
            z_pos_aug = model.encoder(pos_aug)
            contrast_loss = nt_xent_loss(z_pos, z_pos_aug, temperature=0.5)
            
            r_loss = ranking_loss(model.critic, pos_sample, neg_sample)
            total_batch_loss = lambda_factor * r_loss + (1 - lambda_factor) * contrast_loss
            
            optimizer.zero_grad()
            total_batch_loss.backward()
            optimizer.step()
            
            total_loss += total_batch_loss.item()
        
        avg_loss = total_loss / len(human_feedback)
        print(f"[RLHF Epoch {epoch+1}/{num_epochs}] 平均損失: {avg_loss:.4f}")
    print("RLHF 微調完成。")


In [6]:
import torch
import numpy as np

# 假設前面已經定義並建立了 model (TimeSeriesWGAN) 與 rlhf_finetune 函式
# 例如:
# model = build_time_series_model(dataset, latent_dim=100, series_length=128, channels=1, batch_size=64, lr=1e-4)

# ----------------------------
# 模擬人類反饋資料生成函式
# ----------------------------
def generate_simulated_human_feedback(model, num_samples=10):
    """
    根據模型生成兩筆樣本，利用簡單規則模擬 human feedback。
    規則：計算兩個生成樣本的平均絕對值，較大的視為 preferred sample。
    返回一個 list，每個元素為一個 dict：
    {
        "sample1": Tensor (shape: [channels, series_length]),
        "sample2": Tensor (shape: [channels, series_length]),
        "preferred": 1 或 2
    }
    """
    device = model.device
    latent_dim = model.latent_dim
    human_feedback = []

    for _ in range(num_samples):
        # 隨機產生兩個 latent 向量
        z1 = torch.randn(1, latent_dim, device=device)
        z2 = torch.randn(1, latent_dim, device=device)
        # 生成兩筆時間序列資料
        sample1 = model.generator(z1)  # shape: [1, channels, series_length]
        sample2 = model.generator(z2)

        # 模擬一個簡單的評分規則：取平均絕對值較高的視為偏好
        score1 = sample1.abs().mean().item()
        score2 = sample2.abs().mean().item()
        preferred = 1 if score1 > score2 else 2

        # 將批次維度移除，變成 [channels, series_length]
        human_feedback.append({
            "sample1": sample1.squeeze(0),
            "sample2": sample2.squeeze(0),
            "preferred": preferred
        })
    
    return human_feedback

# ----------------------------
# 測試 RLHF 微調是否成功
# ----------------------------
if __name__ == '__main__':
    # 假設先前已建立一個 model，例如使用 dummy 資料建立
    # 如果您還未建立 model，這裡示範使用 dummy 資料：
    dummy_data = np.random.randn(1000, 1, 128).astype(np.float32)
    from torch.utils.data import DataLoader
    dataset = TimeSeriesDataset(dummy_data)
    model = build_time_series_model(dataset, latent_dim=100, series_length=128, channels=1, batch_size=64, lr=1e-4)
    
    # 進行初步訓練，您可以選擇先訓練幾個 epoch
    print("先進行初步訓練...")
    model.train(num_epochs=3)
    
    # 生成模擬的 human feedback 資料
    simulated_feedback = generate_simulated_human_feedback(model, num_samples=10)
    print("模擬的 human feedback 資料已生成，共有", len(simulated_feedback), "筆。")
    
    # 呼叫 RLHF 微調函式
    print("開始 RLHF 微調...")
    rlhf_finetune(model, simulated_feedback, num_epochs=3, lambda_factor=0.5)
    
    # RLHF 微調後，可檢查模型生成結果
    gen_samples = model.generate(5)
    print("RLHF 微調後生成樣本的形狀：", gen_samples.shape)


先進行初步訓練...
訓練開始，使用設備：cuda
[Epoch 1/3] [批次 0/15] [Loss_C: -0.0037] [Loss_G: -0.0033] [Loss_E: 4.0111]
[Epoch 2/3] [批次 0/15] [Loss_C: -0.1024] [Loss_G: 0.0379] [Loss_E: 3.0249]
[Epoch 3/3] [批次 0/15] [Loss_C: -0.0641] [Loss_G: -0.0430] [Loss_E: 2.9759]
訓練完成。
模擬的 human feedback 資料已生成，共有 10 筆。
開始 RLHF 微調...
[RLHF Epoch 1/3] 平均損失: 0.3342
[RLHF Epoch 2/3] 平均損失: 0.3279
[RLHF Epoch 3/3] 平均損失: 0.3213
RLHF 微調完成。
RLHF 微調後生成樣本的形狀： torch.Size([5, 1, 128])


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import random
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import tkinter as tk
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

#############################################
# 1. 時間序列資料集介面
#############################################
class TimeSeriesDataset(Dataset):
    def __init__(self, data):
        """
        熊熊，請確認 data 為 numpy array 或 torch.Tensor，形狀為 (N, channels, series_length)
        """
        if isinstance(data, np.ndarray):
            self.data = torch.from_numpy(data)
        else:
            self.data = data
    
    def __len__(self):
        return self.data.shape[0]
    
    def __getitem__(self, idx):
        return self.data[idx]

#############################################
# 2. 時間序列增強函數 (例如 jittering)
#############################################
def time_series_augmentation(x, noise_std=0.01):
    noise = torch.randn_like(x) * noise_std
    return x + noise

#############################################
# 3. NT-Xent 對比損失函數
#############################################
def nt_xent_loss(z1, z2, temperature=0.5):
    batch_size = z1.shape[0]
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    
    representations = torch.cat([z1, z2], dim=0)  # (2N, latent_dim)
    similarity_matrix = torch.matmul(representations, representations.T)
    sim_exp = torch.exp(similarity_matrix / temperature)
    mask = (~torch.eye(2 * batch_size, 2 * batch_size, dtype=bool, device=z1.device)).float()
    denom = torch.sum(sim_exp * mask, dim=1)
    pos_sim = torch.exp(torch.sum(z1 * z2, dim=1) / temperature)
    pos_sim = torch.cat([pos_sim, pos_sim], dim=0)
    
    loss = -torch.log(pos_sim / denom + 1e-8)
    return loss.mean()

#############################################
# 4. Generator 網路 (針對時間序列生成)
#############################################
class TimeSeriesGenerator(nn.Module):
    def __init__(self, latent_dim=100, series_length=128, channels=1):
        super(TimeSeriesGenerator, self).__init__()
        self.latent_dim = latent_dim
        self.series_length = series_length
        self.channels = channels
        self.hidden_dim = 128
        
        self.fc = nn.Linear(latent_dim, self.hidden_dim * (series_length // 4))
        self.deconv = nn.Sequential(
            nn.ConvTranspose1d(self.hidden_dim, self.hidden_dim // 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(self.hidden_dim // 2),
            nn.ReLU(True),
            nn.ConvTranspose1d(self.hidden_dim // 2, channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )
    
    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, self.hidden_dim, self.series_length // 4)
        x = self.deconv(x)
        return x

#############################################
# 5. Critic 網路 (WGAN 中用於估計 Wasserstein 距離)
#############################################
class TimeSeriesCritic(nn.Module):
    def __init__(self, series_length=128, channels=1):
        super(TimeSeriesCritic, self).__init__()
        self.series_length = series_length
        self.channels = channels
        self.hidden_dim = 128
        
        self.conv = nn.Sequential(
            nn.Conv1d(channels, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(64, self.hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.fc = nn.Linear(self.hidden_dim * (series_length // 4), 1)
    
    def forward(self, x):
        features = self.conv(x)
        features = features.view(features.size(0), -1)
        out = self.fc(features)
        return out

#############################################
# 6. Encoder 網路 (依 TS-TCC 將時間序列映射至隱向量空間)
#############################################
class TimeSeriesEncoder(nn.Module):
    def __init__(self, series_length=128, channels=1, latent_dim=100):
        super(TimeSeriesEncoder, self).__init__()
        self.series_length = series_length
        self.channels = channels
        self.hidden_dim = 128
        
        self.conv = nn.Sequential(
            nn.Conv1d(channels, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, self.hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True)
        )
        self.fc = nn.Linear(self.hidden_dim * (series_length // 4), latent_dim)
    
    def forward(self, x):
        features = self.conv(x)
        features = features.view(features.size(0), -1)
        z = self.fc(features)
        return z

#############################################
# 7. 整合 WGAN 與 TS-TCC 的模型類別
#############################################
class TimeSeriesWGAN:
    def __init__(self, dataset, latent_dim=100, series_length=128, channels=1, batch_size=64, lr=1e-4,
                 weight_clip=0.01, critic_iter=5, device=None):
        self.device = device if device is not None else ('cuda' if torch.cuda.is_available() else 'cpu')
        self.latent_dim = latent_dim
        self.series_length = series_length
        self.channels = channels
        self.batch_size = batch_size
        self.lr = lr
        self.weight_clip = weight_clip
        self.critic_iter = critic_iter
        
        self.generator = TimeSeriesGenerator(latent_dim, series_length, channels).to(self.device)
        self.critic = TimeSeriesCritic(series_length, channels).to(self.device)
        self.encoder = TimeSeriesEncoder(series_length, channels, latent_dim).to(self.device)
        
        self.dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
        
        self.optim_C = optim.RMSprop(self.critic.parameters(), lr=lr)
        self.optim_G = optim.RMSprop(self.generator.parameters(), lr=lr)
        self.optim_E = optim.Adam(self.encoder.parameters(), lr=lr)
    
    def train(self, num_epochs=100):
        print(f"訓練開始，使用設備：{self.device}")
        for epoch in range(num_epochs):
            for i, real_data in enumerate(self.dataloader):
                real_data = real_data.to(self.device)
                batch_size = real_data.size(0)
                
                for _ in range(self.critic_iter):
                    z = torch.randn(batch_size, self.latent_dim, device=self.device)
                    fake_data = self.generator(z).detach()
                    
                    critic_real = self.critic(real_data)
                    critic_fake = self.critic(fake_data)
                    
                    loss_C = -torch.mean(critic_real) + torch.mean(critic_fake)
                    
                    self.optim_C.zero_grad()
                    loss_C.backward()
                    self.optim_C.step()
                    
                    for p in self.critic.parameters():
                        p.data.clamp_(-self.weight_clip, self.weight_clip)
                
                z = torch.randn(batch_size, self.latent_dim, device=self.device)
                fake_data = self.generator(z)
                loss_G = -torch.mean(self.critic(fake_data))
                
                self.optim_G.zero_grad()
                loss_G.backward()
                self.optim_G.step()
                
                aug1 = time_series_augmentation(real_data)
                aug2 = time_series_augmentation(real_data)
                
                z1 = self.encoder(aug1)
                z2 = self.encoder(aug2)
                
                loss_E = nt_xent_loss(z1, z2, temperature=0.5)
                
                self.optim_E.zero_grad()
                loss_E.backward()
                self.optim_E.step()
                
                if i % 50 == 0:
                    print(f"[Epoch {epoch+1}/{num_epochs}] [批次 {i}/{len(self.dataloader)}] "
                          f"[Loss_C: {loss_C.item():.4f}] [Loss_G: {loss_G.item():.4f}] [Loss_E: {loss_E.item():.4f}]")
        print("訓練完成。")
    
    def generate(self, n_samples):
        self.generator.eval()
        with torch.no_grad():
            z = torch.randn(n_samples, self.latent_dim, device=self.device)
            samples = self.generator(z)
        self.generator.train()
        return samples.cpu()
    
    def get_latent(self, x):
        self.encoder.eval()
        with torch.no_grad():
            latent = self.encoder(x.to(self.device))
        self.encoder.train()
        return latent.cpu()
    
    def get_model(self):
        return {
            "generator": self.generator,
            "critic": self.critic,
            "encoder": self.encoder
        }

#############################################
# 8. 模型建構介面函數
#############################################
def build_time_series_model(dataset, latent_dim=100, series_length=128, channels=1, batch_size=64, lr=1e-4,
                            weight_clip=0.01, critic_iter=5, device=None):
    model = TimeSeriesWGAN(dataset, latent_dim=latent_dim, series_length=series_length, channels=channels,
                           batch_size=batch_size, lr=lr, weight_clip=weight_clip, critic_iter=critic_iter,
                           device=device)
    return model

#############################################
# 9. Ranking Loss 與 RLHF 微調函數
#############################################
def ranking_loss(critic, preferred, non_preferred):
    score_pref = critic(preferred)
    score_nonpref = critic(non_preferred)
    loss = -torch.log(torch.sigmoid(score_pref - score_nonpref) + 1e-8)
    return loss.mean()

def rlhf_finetune(model, human_feedback, num_epochs=5, lambda_factor=0.5):
    """
    熊熊，進行 RLHF 微調：
    - model: TimeSeriesWGAN 實例
    - human_feedback: list of dict，每筆包含 "sample1", "sample2", "preferred"
    - num_epochs: 微調的 epoch 數
    - lambda_factor: RL 與對比損失權重
    """
    device = model.device
    optimizer = optim.Adam(list(model.generator.parameters()) + 
                           list(model.critic.parameters()) + 
                           list(model.encoder.parameters()), lr=model.lr)
    
    for epoch in range(num_epochs):
        total_loss = 0.0
        for fb in human_feedback:
            # 為避免梯度版本問題，先 detach 並 clone
            sample1 = fb["sample1"].detach().clone().to(device).unsqueeze(0)
            sample2 = fb["sample2"].detach().clone().to(device).unsqueeze(0)
            if fb["preferred"] == 1:
                pos_sample = sample1
                neg_sample = sample2
            elif fb["preferred"] == 2:
                pos_sample = sample2
                neg_sample = sample1
            else:
                raise ValueError("preferred 欄位必須為 1 或 2")
            
            # 增強正樣本
            pos_aug = time_series_augmentation(pos_sample)
            z_pos = model.encoder(pos_sample)
            z_pos_aug = model.encoder(pos_aug)
            contrast_loss = nt_xent_loss(z_pos, z_pos_aug, temperature=0.5)
            
            r_loss = ranking_loss(model.critic, pos_sample, neg_sample)
            total_batch_loss = lambda_factor * r_loss + (1 - lambda_factor) * contrast_loss
            
            optimizer.zero_grad()
            total_batch_loss.backward()
            optimizer.step()
            
            total_loss += total_batch_loss.item()
        
        avg_loss = total_loss / len(human_feedback)
        print(f"[RLHF Epoch {epoch+1}/{num_epochs}] 平均損失: {avg_loss:.4f}")
    print("RLHF 微調完成。")

#############################################
# 10. 模擬 Human Feedback 生成函式
#############################################
def generate_simulated_human_feedback(model, num_samples=10):
    """
    利用模型生成兩筆樣本，並以簡單規則模擬 human feedback
    """
    device = model.device
    latent_dim = model.latent_dim
    human_feedback = []
    for _ in range(num_samples):
        z1 = torch.randn(1, latent_dim, device=device)
        z2 = torch.randn(1, latent_dim, device=device)
        sample1 = model.generator(z1)
        sample2 = model.generator(z2)
        
        score1 = sample1.abs().mean().item()
        score2 = sample2.abs().mean().item()
        preferred = 1 if score1 > score2 else 2
        
        human_feedback.append({
            "sample1": sample1.squeeze(0),
            "sample2": sample2.squeeze(0),
            "preferred": preferred
        })
    return human_feedback

#############################################
# 11. Human Preference UI (簡單示範)
#############################################
def show_preference_ui(sample1, sample2):
    feedback = {}
    sample1_np = sample1.squeeze().detach().cpu().numpy()
    sample2_np = sample2.squeeze().detach().cpu().numpy()
    
    root = tk.Tk()
    root.title("請選擇較符合偏好的時間序列")
    
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    axs[0].plot(sample1_np)
    axs[0].set_title("樣本 1")
    axs[1].plot(sample2_np)
    axs[1].set_title("樣本 2")
    
    canvas = FigureCanvasTkAgg(fig, master=root)
    canvas.draw()
    canvas.get_tk_widget().pack()
    
    def choose_sample(choice):
        feedback["sample1"] = sample1
        feedback["sample2"] = sample2
        feedback["preferred"] = choice
        root.destroy()
    
    btn_frame = tk.Frame(root)
    btn_frame.pack()
    btn1 = tk.Button(btn_frame, text="選擇樣本 1", command=lambda: choose_sample(1))
    btn1.pack(side="left", padx=20, pady=10)
    btn2 = tk.Button(btn_frame, text="選擇樣本 2", command=lambda: choose_sample(2))
    btn2.pack(side="left", padx=20, pady=10)
    
    root.mainloop()
    return feedback

#############################################
# 12. 測試主程式 (整合示範)
#############################################
if __name__ == '__main__':
    # 建立 dummy 資料集，形狀 (1000, 1, 128)
    dummy_data = np.random.randn(1000, 1, 128).astype(np.float32)
    dataset = TimeSeriesDataset(dummy_data)
    
    # 建立模型
    model = build_time_series_model(dataset, latent_dim=100, series_length=128, channels=1, batch_size=64, lr=1e-4)
    
    # 先進行初步訓練
    print("先進行初步訓練...")
    model.train(num_epochs=3)
    
    # 模擬生成 human feedback 資料
    simulated_feedback = generate_simulated_human_feedback(model, num_samples=10)
    print("模擬的 human feedback 資料已生成，共有", len(simulated_feedback), "筆。")
    
    # 進行 RLHF 微調
    print("開始 RLHF 微調...")
    rlhf_finetune(model, simulated_feedback, num_epochs=3, lambda_factor=0.5)
    
    # 測試生成
    gen_samples = model.generate(5)
    print("RLHF 微調後生成樣本的形狀：", gen_samples.shape)


先進行初步訓練...
訓練開始，使用設備：cuda
[Epoch 1/3] [批次 0/15] [Loss_C: -0.0026] [Loss_G: 0.0088] [Loss_E: 4.1054]
[Epoch 2/3] [批次 0/15] [Loss_C: -0.0887] [Loss_G: 0.0096] [Loss_E: 3.0128]
[Epoch 3/3] [批次 0/15] [Loss_C: -0.0867] [Loss_G: -0.0740] [Loss_E: 2.9584]
訓練完成。
模擬的 human feedback 資料已生成，共有 10 筆。
開始 RLHF 微調...
[RLHF Epoch 1/3] 平均損失: 0.3486
[RLHF Epoch 2/3] 平均損失: 0.3462
[RLHF Epoch 3/3] 平均損失: 0.3445
RLHF 微調完成。
RLHF 微調後生成樣本的形狀： torch.Size([5, 1, 128])
